In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 07. Generative AI: Retrieval-Augmented Generation (RAG)

## Algorithm Category
**Type**: Generative AI - RAG  
**Complexity**: Medium  
**Use Case**: Grounding LLM responses with external knowledge

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand RAG architecture and why it's important
- Understand the difference between RAG and standard generation
- Create vector stores with FAISS for document retrieval
- Chunk documents appropriately for retrieval
- Generate embeddings for documents and queries
- Retrieve relevant documents for a query
- Format context for LLM generation
- Build a complete RAG pipeline

## Historical Context

RAG was introduced by Meta (Facebook) AI Research in 2020:
- Lewis, P., et al. (2020): "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"
- Combines dense passage retrieval with sequence-to-sequence generation
- Allows models to access external knowledge without fine-tuning
- Foundation for many modern AI applications

**Key Papers/References:**
- Lewis, P., et al. (2020). "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"
- FAISS: Facebook AI Similarity Search (efficient vector search)

## What is RAG?

**RAG (Retrieval-Augmented Generation)** combines information retrieval with text generation. Instead of relying solely on the model's training data, RAG:
1. Retrieves relevant documents from a knowledge base
2. Provides these documents as context to the LLM
3. Generates responses grounded in the retrieved information

### Why RAG?

**Problems with Standard LLMs:**
- Limited to training data (may be outdated)
- Can't access private/internal documents
- May hallucinate (make up information)
- No way to verify sources

**RAG Benefits:**
- ✅ Access to up-to-date information
- ✅ Can use private/internal documents
- ✅ Grounded in retrieved sources (less hallucination)
- ✅ Can cite sources (transparency)
- ✅ No fine-tuning needed (works with any LLM)

### Key Concepts

**Document Chunking**: Splitting documents into smaller pieces
- Large documents → Smaller chunks (e.g., 500 tokens each)
- Overlap between chunks (preserves context)
- Enables precise retrieval (find relevant sections)

**Embeddings**: Vector representations of text
- Documents → Embeddings (stored in vector store)
- Query → Embedding (used for search)
- Similarity search: Find documents with similar embeddings

**Vector Store**: Database of document embeddings
- FAISS: Fast similarity search
- Stores embeddings for fast retrieval
- Supports approximate nearest neighbor search

**Retrieval**: Finding relevant documents
- Query embedding compared to document embeddings
- Top-K most similar documents retrieved
- Ranked by similarity score

**Context Formatting**: Preparing retrieved docs for LLM
- Combine retrieved documents
- Format as context for prompt
- Include in LLM prompt before generation

### When to Use RAG

✅ **Good for:**
- Question answering over documents
- Chatbots with knowledge bases
- When you need up-to-date information
- Private/internal document access
- When you need source citations
- Domain-specific applications
- When fine-tuning is not feasible

❌ **Not ideal for:**
- General knowledge questions (LLM already knows)
- Very simple queries (overhead not worth it)
- When all information fits in context window
- Real-time applications with strict latency requirements
- When documents change very frequently (retraining needed)


## Theory & Mechanics

### What is RAG?

RAG combines retrieval with generation in a two-stage process:

**Stage 1: Retrieval**
1. User query → Embedding
2. Compare with document embeddings
3. Retrieve top-K most similar documents
4. Rank by relevance

**Stage 2: Generation**
1. Format retrieved documents as context
2. Combine: Context + Query → LLM prompt
3. LLM generates response using context
4. Response is grounded in retrieved information

### RAG Pipeline

```
Documents
    ↓
Chunking (split into pieces)
    ↓
Embedding (convert to vectors)
    ↓
Vector Store (FAISS)
    ↓
Query → Embedding
    ↓
Retrieval (find similar documents)
    ↓
Context Formatting
    ↓
LLM Generation (with context)
    ↓
Final Response
```

### Document Chunking Strategies

**Fixed-Size Chunks:**
- Split into equal-sized pieces (e.g., 500 tokens)
- Simple but may break sentences/paragraphs

**Sentence-Based Chunks:**
- Split at sentence boundaries
- Preserves sentence integrity
- Better for semantic understanding

**Overlapping Chunks:**
- Chunks overlap (e.g., 100 tokens)
- Preserves context across boundaries
- Reduces information loss at boundaries

### Retrieval Methods

**Dense Retrieval (Embeddings):**
- Convert text to dense vectors
- Use cosine similarity
- Fast and effective

**Sparse Retrieval (Keywords):**
- Use TF-IDF or BM25
- Keyword-based matching
- Good for exact matches

**Hybrid Retrieval:**
- Combine dense + sparse
- Best of both worlds
- More accurate but slower


## Installation & Setup


## Implementation


In [ ]:
# ============================================
# SETTING UP PYTHON PATH: Accessing Project Modules
# ============================================

# Add project root to Python path for imports
# This allows us to import modules from the src/ directory
import sys  # sys: System-specific parameters and functions
from pathlib import Path  # Path: Object-oriented filesystem paths

# Get the project root (two levels up from this notebook)
# notebooks/generative_ai/ -> notebooks/ -> project_root/
project_root = Path().resolve().parent.parent
# Path().resolve(): Get current directory (notebooks/generative_ai/)
# .parent: Go up one level (notebooks/)
# .parent: Go up another level (project_root/)

# Add project root to Python path if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    # sys.path: List of directories Python searches for modules
    # .insert(0, ...): Add to beginning (highest priority)
    # This allows: from src.llm.rag_utils import ...

# ============================================
# IMPORTING RAG UTILITIES: Retrieval-Augmented Generation Functions
# ============================================

# Our custom RAG utilities
from src.llm.rag_utils import chunk_documents, create_vector_store, retrieve_relevant_docs, format_rag_context
# chunk_documents(): Split documents into smaller chunks
#   - Takes large documents and splits them
#   - Configurable chunk size and overlap
#   - Preserves context across boundaries
#   - Returns list of document chunks
# create_vector_store(): Create FAISS vector store from documents
#   - Generates embeddings for all chunks
#   - Stores embeddings in FAISS index
#   - Enables fast similarity search
#   - Returns vector store object
# retrieve_relevant_docs(): Find relevant documents for a query
#   - Embeds the query
#   - Searches vector store for similar documents
#   - Returns top-K most relevant chunks
#   - Ranked by similarity score
# format_rag_context(): Format retrieved documents for LLM
#   - Combines retrieved chunks
#   - Formats as context for prompt
#   - Includes metadata (sources, scores)
#   - Returns formatted context string

print("RAG utilities imported")  # Confirm imports successful

# ============================================
# KEY CONCEPTS
# ============================================

# RAG (Retrieval-Augmented Generation):
# 1. Retrieval: Find relevant documents from knowledge base
# 2. Augmentation: Add retrieved docs as context
# 3. Generation: LLM generates response using context
#
# Pipeline:
# Documents → Chunking → Embedding → Vector Store
# Query → Embedding → Retrieval → Context → LLM → Response
#
# Benefits:
# - Access to up-to-date information
# - Can use private documents
# - Grounded in sources (less hallucination)
# - Can cite sources
# - No fine-tuning needed
#
# Key Components:
# - Document chunking: Split large docs into searchable pieces
# - Embeddings: Vector representations for similarity search
# - Vector store: Fast retrieval (FAISS)
# - Context formatting: Prepare docs for LLM
# - LLM generation: Generate response with context

## Validation & Testing


In [ ]:
# Validation
print("Validation")

## Performance Benchmarking


In [ ]:
# Performance
print("Performance")

## Summary & Key Takeaways

- Key concept 1
- Key concept 2
- Key concept 3
